# Recommendations

Purchase history with ratings, from `data/sample_history.json`: made-up data shaped like the item service's `History` (with its `Item`) and `Review` models. Regenerate or resize it with `python3 data/make_sample_history.py [count]`.

In [1]:
import json
from pathlib import Path

import pandas as pd

raw = json.loads(Path("data/sample_history.json").read_text(encoding="utf-8"))
print(f"{len(raw)} purchases, {sum(h['review'] is not None for h in raw)} rated")

60 purchases, 48 rated


## One row per purchase

The item's and the review's fields flattened next to the history's; `score` and `content` are empty for purchases not rated yet.

In [2]:
history = pd.json_normalize(raw).drop(columns=["item.id", "review", "review.id", "review.history_id"], errors="ignore")
history.columns = [c.split(".", 1)[1] if c.startswith(("item.", "review.")) else c for c in history.columns]
history = history.rename(columns={"id": "history_id"})
for col in ["time", "created_at", "deadline_at", "buy_at", "schedule"]:
    history[col] = pd.to_datetime(history[col])
history["score"] = history["score"].astype("Int64")

history[["time", "owner", "item_type", "item_name", "shop_name", "score", "content"]].head(10)

,time,owner,item_type,item_name,shop_name,score,content
0,2026-11-24 22:17:00+07:00,lan,food_and_drink,Trà đào cam sả,Highlands Coffee,<NA>,NaN
1,2026-11-20 19:43:20+07:00,lan,food_and_drink,Trà sữa trân châu đường đen,Phúc Long,5,"Hoàn hảo, không chê được gì"
2,2026-11-10 07:09:07+07:00,lan,food_and_drink,Cà phê muối,Cà Phê Muối Chú Long,3,"Bình thường, không có gì đặc biệt"
3,2026-11-09 15:45:20+07:00,minh,restaurant,Lẩu Thái Tomyum,None,5,"Hoàn hảo, không chê được gì"
4,2026-11-07 16:26:20+07:00,lan,clothes,Túi tote canvas,Shopee,4,"Ngon, sẽ quay lại"
5,2026-10-30 07:21:44+07:00,lan,food_and_drink,Cà phê muối,Cà Phê Muối Chú Long,4,"Ngon, sẽ quay lại"
6,2026-10-26 21:07:22+07:00,lan,restaurant,Bún chả Hương Liên,None,4,"Ngon, sẽ quay lại"
7,2026-10-17 21:16:21+07:00,lan,food_and_drink,Trà sữa trân châu đường đen,Phúc Long,<NA>,NaN
8,2026-10-17 11:17:43+07:00,minh,clothes,Áo hoodie nỉ,Uniqlo Đồng Khởi,4,"Ngon, sẽ quay lại"
9,2026-10-15 11:14:18+07:00,lan,restaurant,Sushi Hokkaido Sachi,None,3,"Ổn, nhưng có chỗ khác ngon hơn"


## Ratings

The rated purchases only, and each person's average score per item: the user–item matrix a recommender starts from.

In [3]:
ratings = history.dropna(subset=["score"])[["owner", "item_type", "item_name", "score"]]
ratings.groupby(["owner", "item_type"])["score"].agg(["count", "mean"]).round(2)

count  mean
owner item_type                  
lan   clothes            11  3.73
      food_and_drink     12  4.08
      others              2   4.0
      restaurant          6   4.0
minh  clothes             8  3.75
      food_and_drink      5   4.0
      others              2   2.5
      restaurant          2   5.0

In [4]:
matrix = ratings.pivot_table(index="owner", columns="item_name", values="score", aggfunc="mean").round(1)
matrix

item_name,Bánh flan caramel,Bánh tráng trộn,Bún chả Hương Liên,Cà phê muối,Giày Air Force 1,Gogi House,Gấu bông Capybara,Lẩu Thái Tomyum,Matcha latte,Nến thơm vanilla,...,Son kem lì,Sushi Hokkaido Sachi,Trà sữa trân châu đường đen,Trà đào cam sả,Túi tote canvas,Váy hoa nhí,Áo hoodie nỉ,Áo sơ mi linen,Áo thun oversize,Ốp lưng điện thoại
owner,,,,,,,,,,,,,,,,,,,,,
lan,<NA>,3.5,4.0,3.5,3.0,5.0,<NA>,3.0,4.5,4.0,...,<NA>,3.0,4.8,<NA>,4.0,4.0,4.5,3.0,4.0,4.0
minh,3.0,<NA>,<NA>,<NA>,4.5,<NA>,3.0,5.0,4.0,<NA>,...,2.0,<NA>,4.0,4.0,2.0,<NA>,4.0,3.0,4.0,<NA>
